# Traffic Accident Data Cleaning and Preprocessing

This notebook documents the **actual data-cleaning workflow** used for the Traffic Accidents Analytics project. It loads `traffic_accidents_raw.csv`, checks the dataset, removes duplicate records/IDs, standardizes text/date/time fields, validates numeric fields, and exports `traffic_accidents_cleaned.csv`.

**Source files:** `traffic_accidents_raw.csv` → `traffic_accidents_cleaned.csv`


## 1. Import libraries and define file paths

The cleaning script uses pandas for data processing and `Path` for portable file paths.

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_FILE = Path("traffic_accidents_raw.csv")
OUTPUT_FILE = Path("traffic_accidents_cleaned.csv")

print(f"Input file: {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")

## 2. Load and inspect the raw dataset

The repository's raw CSV currently contains a merge-conflict marker at the very beginning (`<<<<<<< HEAD`). The notebook detects that marker and removes it from a temporary in-memory copy before parsing. This prevents the marker from becoming a false column name and allows the actual CSV data to be inspected.

In [ ]:
raw_text = INPUT_FILE.read_text(encoding="utf-8")

had_conflict_marker = raw_text.startswith("<<<<<<< HEAD")

if had_conflict_marker:
    lines = raw_text.splitlines()
    if lines and lines[0].strip() == "<<<<<<< HEAD":
        lines = lines[1:]
    raw_text = "\n".join(lines) + "\n"
    print("Merge-conflict marker detected and removed before parsing.")
else:
    print("No merge-conflict marker detected.")

from io import StringIO
df = pd.read_csv(StringIO(raw_text))

print(f"Raw dataset loaded successfully.")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
display(df.head())

## 3. Dataset structure and missing-value check

The dataset contains accident identifiers, date/time information, location, severity, weather, road condition, vehicle type, vehicle count, injuries, fatalities, and accident cause.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values by column:")
display(df.isna().sum().to_frame("Missing Values"))

print(f"\nTotal missing values: {df.isna().sum().sum()}")

## 4. Remove completely duplicated rows

Duplicate rows are identified and removed so that the same accident record is not counted more than once.

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f"Duplicate rows found: {duplicate_rows}")

df = df.drop_duplicates()
print(f"Rows after removing duplicate rows: {len(df)}")

## 5. Remove duplicate Accident IDs

`Accident_ID` is treated as the unique identifier for each accident. If the same ID occurs more than once, the first occurrence is retained.

In [ ]:
duplicate_ids = df["Accident_ID"].duplicated().sum()
print(f"Duplicate Accident IDs found: {duplicate_ids}")

df = df.drop_duplicates(subset="Accident_ID", keep="first")
print(f"Rows after removing duplicate Accident IDs: {len(df)}")

## 6. Clean text columns

Leading and trailing whitespace is removed from the identifier, location, categorical, and cause fields.

In [ ]:
text_columns = [
    "Accident_ID",
    "City",
    "State",
    "Accident_Severity",
    "Weather_Condition",
    "Road_Condition",
    "Vehicle_Type",
    "Accident_Cause"
]

for column in text_columns:
    df[column] = df[column].astype(str).str.strip()

print("Text columns standardized:")
print(", ".join(text_columns))

## 7. Standardize the Date column

Dates are converted to pandas datetime values with invalid values coerced to `NaT`. The final format is `YYYY-MM-DD`.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
invalid_dates = df["Date"].isna().sum()

print(f"Invalid dates found: {invalid_dates}")

df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
print("Date format standardized to YYYY-MM-DD.")

## 8. Standardize the Time column

Times are parsed using the expected 24-hour `HH:MM` format. Invalid values are converted to missing values.

In [ ]:
df["Time"] = pd.to_datetime(
    df["Time"],
    format="%H:%M",
    errors="coerce"
).dt.strftime("%H:%M")

invalid_times = df["Time"].isna().sum()
print(f"Invalid times found: {invalid_times}")
print("Time format standardized to HH:MM.")

## 9. Validate numerical columns

`Vehicles_Involved`, `Injuries`, and `Fatalities` are converted to numeric values. Non-numeric values are coerced to missing values so they can be identified.

In [ ]:
numeric_columns = [
    "Vehicles_Involved",
    "Injuries",
    "Fatalities"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Numeric columns converted successfully.")
display(df[numeric_columns].describe())

## 10. Check for invalid negative values

Counts such as vehicles involved, injuries, and fatalities cannot logically be negative. Negative values are therefore checked before export.

In [ ]:
negative_counts = {}

for column in numeric_columns:
    negative_values = (df[column] < 0).sum()
    negative_counts[column] = int(negative_values)
    print(f"Negative values in {column}: {negative_values}")

## 11. Final validation before export

The cleaned dataset is checked for its final dimensions, remaining missing values, duplicate IDs, and date/time validity.

In [ ]:
print("Final validation")
print("-" * 40)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Remaining missing values: {df.isna().sum().sum()}")
print(f"Remaining duplicate rows: {df.duplicated().sum()}")
print(f"Remaining duplicate Accident IDs: {df['Accident_ID'].duplicated().sum()}")
print(f"Invalid dates after cleaning: {pd.to_datetime(df['Date'], errors='coerce').isna().sum()}")
print(f"Invalid times after cleaning: {pd.to_datetime(df['Time'], format='%H:%M', errors='coerce').isna().sum()}")


## 12. Export the cleaned dataset

The final cleaned dataframe is saved as `traffic_accidents_cleaned.csv`, matching the project deliverable and the existing cleaning script.

In [ ]:
df.to_csv(OUTPUT_FILE, index=False)

print("Cleaning completed successfully.")
print(f"Final rows: {df.shape[0]}")
print(f"Final columns: {df.shape[1]}")
print(f"Cleaned dataset saved as: {OUTPUT_FILE}")

## 13. Cleaning results

Based on the current project dataset, the cleaning workflow produces a **500-row, 13-column** cleaned dataset. The accident IDs run from `TA1001` through `TA1500`. The current records contain no expected duplicate Accident IDs, invalid date/time values, or negative values in the validated numeric fields.

### Main transformations performed
- Removed the raw-file merge-conflict marker before parsing.
- Checked for missing values.
- Removed completely duplicated rows.
- Removed duplicate `Accident_ID` values while keeping the first record.
- Trimmed whitespace from text/categorical columns.
- Standardized dates to `YYYY-MM-DD`.
- Standardized times to 24-hour `HH:MM`.
- Converted vehicle, injury, and fatality fields to numeric values.
- Checked numeric fields for negative values.
- Exported the cleaned dataset to `traffic_accidents_cleaned.csv`.

> **Note:** Run all cells in this notebook after cloning/downloading the repository to reproduce the checks and generated results directly from the project CSV files.